In [27]:
import pandas as pd
import os
import json
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from collections import defaultdict

## Concatenation

In [20]:
files = [f for f in os.listdir("../results/cancer_mouse") if f.endswith(".csv")]

dfs = []
for f in files:
    df = pd.read_csv(os.path.join("../results/cancer_mouse", f))
    df['source'] = f.replace("__preds.csv", "")
    dfs.append(df)

all_preds = pd.concat(dfs, ignore_index=True)
all_preds.to_csv(
    os.path.join("../results/cancer_mouse", "all_cancer_predictions.csv"),
    index=False
)
print("Saved all_cancer_predictions.csv")

Saved all_cancer_predictions.csv


## Analysis

In [21]:
# Load and merge
preds = pd.read_csv("../results/cancer_mouse/all_cancer_predictions.csv")
stats = pd.read_csv("../data/disease_model_stats.csv")
with open("../data/cancer_children.json") as f:
    children = json.load(f)

preds = preds[preds['related_words'].notna()]
stats['ID_'] = stats['ID'].str.replace(":", "_")
merged = (
    pd.merge(preds, stats[['ID_', 'name']],
             left_on='source', right_on='ID_', how='left')
    .drop(columns=['ID_'])
)

# Subset for parent + children of MONDO_0004992
parent = "MONDO_0004992"
terms = children['_embedded']['terms']
child_ids = [t['short_form'] for t in terms if 'cancer' in t['label']]
models = child_ids + [parent]
sub = merged[merged['source'].isin(models)]

# Counts per subtype
counts = sub.groupby('name').size().reset_index(name='count').sort_values('count', ascending=False)

### Pie chart

In [7]:
plt.figure()
circle = plt.Circle((0,0), 0.7, color='white')
plt.pie(counts['count'], labels=counts['name'])
plt.gca().add_artist(circle)
plt.title("Cancer subtype distribution")
plt.show()

### Sankey Diagram

In [9]:
# Wait for debug
labels = ["cancer"] + counts['name'].tolist()
source_idxs = [0] * len(counts)
target_idxs = list(range(1, len(counts)+1))
values = counts['count'].tolist()

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20,
              line=dict(color="black", width=0.5),
              label=labels),
    link=dict(source=source_idxs, target=target_idxs, value=values)
)])
fig.update_layout(title_text="GSEs Predicted as Cancer Subtypes", font_size=12)
fig.show()

## Question

In [22]:
merged

,ID,prob,log2(prob/prior),related_words,source,name
0,GSE92891,0.894172,4.363214,lung,MONDO_0000376,respiratory system cancer
1,GSE83991,0.802160,4.206552,"epithelial,lung",MONDO_0000376,respiratory system cancer
2,GSE51144,0.676622,3.961011,"cancer,lung,nonsmall",MONDO_0000376,respiratory system cancer
3,GSE84447,0.658007,3.920765,"cancer,lung",MONDO_0000376,respiratory system cancer
4,GSE99235,0.593594,3.772139,"cell,lung",MONDO_0000376,respiratory system cancer
...,...,...,...,...,...,...
44166,GSE43366,0.102958,-3.541654,progression,MONDO_0045024,cancer or benign tumor
44167,GSE56284,0.095789,-3.645777,profiling,MONDO_0045024,cancer or benign tumor
44168,GSE68715,0.095587,-3.648814,"analysis,cells,primary,rnaseq",MONDO_0045024,cancer or benign tumor
44169,GSE93412,0.077718,-3.947386,profiles,MONDO_0045024,cancer or benign tumor


In [49]:
CANCER = "MONDO_0004992"
threshold = 0.8
results = pd.DataFrame(columns=['GSE', 'generic', 'specific'])

for i in range(len(merged)):
    prob = merged.loc[i, 'prob']
    source = merged.loc[i, 'source']
    gse_id = merged.loc[i, 'ID']

    if prob < threshold:
        continue

    has_generic = (source == CANCER)
    has_specific = (source in models and source != CANCER)

    results.loc[len(results)] = [gse_id, has_generic, has_specific]

In [50]:
# Aggregate to get unique GSEs
results_unique = results.groupby('GSE', as_index=False).agg({
    'generic': 'any',
    'specific': 'any'
})

# Question 1: Generic or Specific
mapped_to_cancer = results_unique[(results_unique['generic']) | (results_unique['specific'])]
print(f"Total GSEs mapped to cancer (generic or specific): {len(mapped_to_cancer)}")

# Question 2: Specific only
specific_only = results_unique[(results_unique['specific']) & (~results_unique['generic'])]
print(f"Specific cancer only (missing generic): {len(specific_only)}")

# Question 3: Generic Only
generic_only = results_unique[(results_unique['generic']) & (~results_unique['specific'])]

# Bonus: Both
both = results_unique[(results_unique['generic']) & (results_unique['specific'])]

print(f"Generic only GSEs: {len(generic_only)}")
print(f"Both generic and specific: {len(both)}")


Total GSEs mapped to cancer (generic or specific): 51
Specific cancer only (missing generic): 3
Generic only GSEs: 42
Both generic and specific: 6


In [48]:
merged['ID'].nunique()

3886

In [53]:
merged[merged['source'] == 'MONDO_0004953']

,ID,prob,log2(prob/prior),related_words,source,name
